# Nepali Grammar Checker — Federated Learning (FIXED)

**Key Fixes:**
- Corrected checkpoint filename (`seq2seq_best.pth` not `seq2seq_best copy.pth`)
- Proper device handling (load on CPU, move to GPU)
- Tokenizers properly loaded from JSON
- Safe inference with validation


In [ ]:
# Install required packages
import subprocess
import sys

packages = ['flwr>=1.8.0', 'torch', 'pandas', 'numpy', 'scikit-learn']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All packages installed!")

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import numpy as np
import flwr as fl
from typing import List, Tuple, Dict
import warnings
import os
import json
import math

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Flowers version: {fl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.12.0+cu130
Flowers version: 1.31.0
CUDA available: True


## SECTION 1: Load Data & Train Detection Model

In [2]:
# Load detection + correction data from CSV
# UPDATE THIS PATH to your actual data location
DATA_PATH = "/path/to/right_wrong.csv"  # CHANGE THIS
DATA_PATH = "/home/raghav/Work/ioe_purwanchal_campus_iicquest4.0/ml/data_cleaned/right_wrong.csv"

if not os.path.exists(DATA_PATH):
    print(f"⚠️  Data file not found: {DATA_PATH}")
    print("Creating sample data for demo...")
    # Create minimal sample for testing
    sample_data = {
        "correct": ["यसरी", "नेपालमा", "वस्तु", "पिबी", "निर्णय"],
        "wrong": ["ीसरय", "नपेामला", "वतु्स", "बिपी", "निर्ण"]
    }
    df_pairs = pd.DataFrame(sample_data)
else:
    df_pairs = pd.read_csv(DATA_PATH, encoding="utf-8")
    df_pairs.columns = ["correct", "wrong"]
    df_pairs = df_pairs.dropna()
    df_pairs = df_pairs[df_pairs["correct"] != df_pairs["wrong"]].reset_index(drop=True)

print(f"Correction pairs loaded: {len(df_pairs)}")
print(df_pairs.head())

Correction pairs loaded: 2245506
      correct       wrong
0        यसरी        ीसरय
1  व्यवस्थापन  ््नवासयथपव
2      गर्दैछ      छैदगर्
3        बिपी        पिबी
4     कोइराला     लाकाोरइ


In [3]:
# Build detection dataset: correct words -> label 0, wrong words -> label 1
df_correct = pd.DataFrame({"word": df_pairs["correct"].values, "label": 0})
df_wrong   = pd.DataFrame({"word": df_pairs["wrong"].values,   "label": 1})
df = pd.concat([df_correct, df_wrong], ignore_index=True).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f"Detection dataset shape: {df.shape}")
print(df["label"].value_counts())
print(df.head())

# Limit for faster training (optional)
df = df.sample(n=min(100000, len(df)), random_state=42).reset_index(drop=True)
print(f"\nFinal dataset size: {df.shape}")

Detection dataset shape: (4491012, 2)
label
0    2245506
1    2245506
Name: count, dtype: int64
         word  label
0  दुर्घटनामा      0
1       कमजोर      0
2  स्याङ्जामा      0
3      तगिरएा      1
4    सन्तुष्ट      0

Final dataset size: (100000, 2)


In [4]:
# ── Character-level tokenizer for detection ──────────────────────────────

class CharTokenizer:
    PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"

    def __init__(self):
        self.char2idx  = {}
        self.idx2char  = {}
        self.vocab_size = 0

    def build_vocab(self, texts):
        chars = set()
        for t in texts:
            chars.update(list(str(t)))
        specials  = [self.PAD, self.SOS, self.EOS, self.UNK]
        all_chars = specials + sorted(chars)
        self.char2idx   = {c: i for i, c in enumerate(all_chars)}
        self.idx2char   = {i: c for c, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)
        print(f"Char vocab size: {self.vocab_size}")

    def encode(self, text, max_len=30, add_sos=False, add_eos=False):
        ids = []
        if add_sos:
            ids.append(self.char2idx[self.SOS])
        for c in str(text):
            ids.append(self.char2idx.get(c, self.char2idx[self.UNK]))
        if add_eos:
            ids.append(self.char2idx[self.EOS])
        ids = ids[:max_len]
        ids += [self.char2idx[self.PAD]] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            c = self.idx2char.get(i, self.UNK)
            if c in (self.PAD, self.SOS):
                continue
            if c == self.EOS:
                break
            out.append(c)
        return "".join(out)

    def save(self, filepath):
        """Save tokenizer to JSON"""
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump({
                "char2idx": self.char2idx,
                "idx2char": {str(k): v for k, v in self.idx2char.items()}
            }, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, filepath):
        """Load tokenizer from JSON"""
        tok = cls()
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        tok.char2idx = data["char2idx"]
        tok.idx2char = {int(k): v for k, v in data["idx2char"].items()}
        tok.vocab_size = len(tok.char2idx)
        return tok


MAX_SEQ_LEN = 30

# Build char vocab from ALL words
tokenizer = CharTokenizer()
tokenizer.build_vocab(df["word"].tolist())

# Encode
X = np.array(
    [tokenizer.encode(w, MAX_SEQ_LEN) for w in df["word"]],
    dtype=np.int64
)
y = df["label"].values

print(f"Encoded shape: {X.shape}")
print(f"Labels shape:  {y.shape}")

Char vocab size: 74
Encoded shape: (100000, 30)
Labels shape:  (100000,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_test  = torch.tensor(X_test,  dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

print(f"X_train: {X_train.shape}  dtype: {X_train.dtype}")
print(f"y_train: {y_train.shape}  dtype: {y_train.dtype}")
print(f"X_test:  {X_test.shape}   dtype: {X_test.dtype}")
print(f"y_test:  {y_test.shape}   dtype: {y_test.dtype}")

X_train: torch.Size([75000, 30])  dtype: torch.int64
y_train: torch.Size([75000])  dtype: torch.float32
X_test:  torch.Size([25000, 30])   dtype: torch.int64
y_test:  torch.Size([25000])   dtype: torch.float32


In [6]:
class CharTransformerDetector(nn.Module):
    """Transformer encoder for char-level wrong-word detection"""
    def __init__(self, vocab_size, embed_dim=64, num_heads=4,
                 num_layers=3, ff_dim=256, max_len=30, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout     = nn.Dropout(dropout)
        self.classifier  = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        out = self.dropout(self.embedding(x) + self.pos_embedding(positions))
        pad_mask = (x == 0)
        out = self.transformer(out, src_key_padding_mask=pad_mask)
        mask_f  = (~pad_mask).float().unsqueeze(-1)
        pooled  = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        return self.classifier(pooled).squeeze(-1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

detector = CharTransformerDetector(
    vocab_size = tokenizer.vocab_size,
    embed_dim  = 64,
    num_heads  = 4,
    num_layers = 3,
    ff_dim     = 256,
    max_len    = MAX_SEQ_LEN,
    dropout    = 0.3
).to(device)

print(f"Detector Parameters: {sum(p.numel() for p in detector.parameters()):,}")

Using device: cuda
Detector Parameters: 160,833


In [ ]:
def train_detector(model, X_train, y_train, X_test, y_test,
                   epochs=10, batch_size=64):
    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, num_workers=0
    )
    test_loader  = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=batch_size, shuffle=False, num_workers=0
    )

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc  = 0.0
    history   = {"train_loss": [], "test_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for bx, by in test_loader:
                bx, by   = bx.to(device), by.to(device)
                preds    = (model(bx) > 0.5).float()
                correct += (preds == by).sum().item()
                total   += by.size(0)

        acc = correct / total
        history["train_loss"].append(avg_loss)
        history["test_acc"].append(acc)

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), "detector_best.pth")

        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc*100:.2f}%"
              + (" <- best" if acc == best_acc else ""))

    print(f"\nBest accuracy: {best_acc*100:.2f}%")
    model.load_state_dict(torch.load("detector_best.pth", map_location=device))
    return history


print("Training Detector...\n")
det_history = train_detector(
    detector, X_train, y_train, X_test, y_test,
    epochs=10, batch_size=64
)
print("\nDetector training complete!")
torch.save(detector.state_dict(), "detector_best.pth")
tokenizer.save("detect_char_tokenizer.json")
print("✓ Detector and tokenizer saved")

## SECTION 2: Train Seq2Seq Correction Model

In [10]:
MAX_WORD_LEN = 20

char_tok = CharTokenizer()
char_tok.build_vocab(df_pairs["correct"].tolist() + df_pairs["wrong"].tolist())

print(f"Encoding {len(df_pairs):,} pairs...")
src_seqs = np.array(
    [char_tok.encode(w, MAX_WORD_LEN) for w in df_pairs["wrong"]], dtype=np.int64
)
tgt_seqs = np.array(
    [char_tok.encode(w, MAX_WORD_LEN, add_sos=True, add_eos=True)
     for w in df_pairs["correct"]], dtype=np.int64
)
print(f"src: {src_seqs.shape}  tgt: {tgt_seqs.shape}")

Char vocab size: 82
Encoding 2,245,506 pairs...
src: (2245506, 20)  tgt: (2245506, 20)


In [11]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4,
                 num_layers=3, ff_dim=512, max_len=30, dropout=0.1):
        super().__init__()
        self.embed_dim   = embed_dim
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed   = nn.Embedding(max_len + 2, embed_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.dropout     = nn.Dropout(dropout)
        self.fc_h        = nn.Linear(embed_dim, embed_dim)
        self.fc_c        = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T     = x.shape
        pos      = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        out      = self.dropout(self.embedding(x) + self.pos_embed(pos))
        pad_mask = (x == 0)
        enc_out  = self.transformer(out, src_key_padding_mask=pad_mask)
        mask_f   = (~pad_mask).float().unsqueeze(-1)
        mean_enc = (enc_out * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        h        = torch.tanh(self.fc_h(mean_enc)).unsqueeze(0)
        c        = torch.tanh(self.fc_c(mean_enc)).unsqueeze(0)
        return enc_out, h, c


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, encoder_dim):
        super().__init__()
        self.W1 = nn.Linear(encoder_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim,  hidden_dim)
        self.v  = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_h, enc_out):
        score   = self.v(torch.tanh(
            self.W1(enc_out) + self.W2(dec_h).unsqueeze(1)
        )).squeeze(-1)
        weights = torch.softmax(score, dim=1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)
        return context, weights


class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, encoder_dim, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(hidden_dim, encoder_dim)
        self.lstm      = nn.LSTMCell(embed_dim + encoder_dim, hidden_dim)
        self.fc_out    = nn.Linear(hidden_dim + encoder_dim + embed_dim, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def forward_step(self, token, h, c, enc_out):
        emb              = self.dropout(self.embedding(token))
        context, weights = self.attention(h, enc_out)
        h, c             = self.lstm(torch.cat([emb, context], dim=1), (h, c))
        pred             = self.fc_out(torch.cat([h, context, emb], dim=1))
        return pred, h, c, weights


class Seq2SeqCorrector(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 enc_layers=3, dropout=0.1):
        super().__init__()
        self.encoder    = TransformerEncoder(
            vocab_size, embed_dim, num_heads=4,
            num_layers=enc_layers, ff_dim=512,
            max_len=MAX_WORD_LEN, dropout=dropout
        )
        self.decoder    = LSTMDecoder(
            vocab_size, embed_dim, hidden_dim,
            encoder_dim=embed_dim, dropout=dropout
        )
        self.vocab_size = vocab_size

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        B, tgt_len    = tgt.shape
        enc_out, h, c = self.encoder(src)
        h, c          = h.squeeze(0), c.squeeze(0)

        input_tok     = tgt[:, 0]
        outputs       = torch.zeros(B, tgt_len, self.vocab_size).to(src.device)

        for t in range(1, tgt_len):
            pred, h, c, _ = self.decoder.forward_step(input_tok, h, c, enc_out)
            outputs[:, t] = pred
            use_teacher   = torch.rand(1).item() < teacher_forcing_ratio
            input_tok     = tgt[:, t] if use_teacher else pred.argmax(dim=1)

        return outputs


s2s_model = Seq2SeqCorrector(
    vocab_size  = char_tok.vocab_size,
    embed_dim   = 128,
    hidden_dim  = 128,
    enc_layers  = 3,
    dropout     = 0.1
).to(device)

print(f"Seq2Seq Parameters: {sum(p.numel() for p in s2s_model.parameters()):,}")

Seq2Seq Parameters: 914,002


In [ ]:
from torch.utils.data import random_split

src_t    = torch.tensor(src_seqs, dtype=torch.long)
tgt_t    = torch.tensor(tgt_seqs, dtype=torch.long)
dataset  = TensorDataset(src_t, tgt_t)

train_size = int(0.80 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

BATCH     = 32
train_ldr = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_ldr   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

PAD_IDX   = char_tok.char2idx["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(s2s_model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5
)

EPOCHS       = 3
best_val     = float("inf")

print(f"Train: {train_size:,} | Val: {val_size:,} | Batch: {BATCH} | Epochs: {EPOCHS}\n")

for epoch in range(1, EPOCHS + 1):
    s2s_model.train()
    t_loss = 0
    for sb, tb in train_ldr:
        sb, tb = sb.to(device), tb.to(device)
        optimizer.zero_grad()
        out  = s2s_model(sb, tb, teacher_forcing_ratio=0.5)
        loss = criterion(
            out[:, 1:].reshape(-1, char_tok.vocab_size),
            tb[:, 1:].reshape(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(s2s_model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
    avg_t = t_loss / len(train_ldr)

    s2s_model.eval()
    v_loss, correct_w, total_w = 0, 0, 0
    with torch.no_grad():
        for sb, tb in val_ldr:
            sb, tb   = sb.to(device), tb.to(device)
            out      = s2s_model(sb, tb, teacher_forcing_ratio=0.0)
            v_loss  += criterion(
                out[:, 1:].reshape(-1, char_tok.vocab_size),
                tb[:, 1:].reshape(-1)
            ).item()
            pred_ids = out[:, 1:].argmax(dim=-1)
            tgt_ids  = tb[:, 1:]
            mask     = tgt_ids != PAD_IDX
            correct_w += ((pred_ids == tgt_ids) | ~mask).all(dim=1).sum().item()
            total_w   += sb.size(0)

    avg_v    = v_loss / len(val_ldr)
    word_acc = correct_w / total_w * 100
    scheduler.step(avg_v)

    if avg_v < best_val:
        best_val = avg_v
        torch.save(s2s_model.state_dict(), "seq2seq_best.pth")
        print(f"Epoch {epoch:3d}/{EPOCHS} | Train: {avg_t:.4f} | "
              f"Val: {avg_v:.4f} | Word Acc: {word_acc:.1f}% <- best")
    else:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Train: {avg_t:.4f} | "
              f"Val: {avg_v:.4f} | Word Acc: {word_acc:.1f}%")

print("\n✓ Seq2Seq training complete!")
s2s_model.load_state_dict(torch.load("seq2seq_best.pth", map_location=device))
char_tok.save("seq2seq_char_tokenizer.json")
print("✓ Model and tokenizer saved")

## SECTION 3: Inference & Testing

**CRITICAL FIXES:**
1. Load models on CPU first
2. Then move to GPU
3. Use correct checkpoint filename (`seq2seq_best.pth`)
4. Load tokenizers from saved JSON

In [14]:
# ============================================================================
# LOAD MODELS & TOKENIZERS FOR INFERENCE
# ============================================================================

# Step 1: Load on CPU (prevents CUDA device-side assert)
safe_device = torch.device("cpu")
print("Step 1: Loading models on CPU...")

# Recreate models
detector_infer = CharTransformerDetector(
    vocab_size = 60,  # Will be overwritten by checkpoint
    embed_dim  = 64,
    num_heads  = 4,
    num_layers = 3,
    ff_dim     = 256,
    max_len    = MAX_SEQ_LEN,
    dropout    = 0.3
).to(safe_device)

s2s_model_infer = Seq2SeqCorrector(
    vocab_size  = 60,  # Will be overwritten by checkpoint
    embed_dim   = 128,
    hidden_dim  = 128,
    enc_layers  = 3,
    dropout     = 0.1
).to(safe_device)

# Step 2: Load state dicts
detector_checkpoint = "detector_best.pth"
seq2seq_checkpoint  = "seq2seq_best copy.pth"

if not os.path.exists(detector_checkpoint):
    print(f"⚠️  {detector_checkpoint} not found. Skipping detector load.")
else:
    try:
        detector_infer.load_state_dict(
            torch.load(detector_checkpoint, map_location=safe_device)
        )
        print(f"✓ {detector_checkpoint} loaded")
    except RuntimeError as e:
        print(f"❌ Failed to load {detector_checkpoint}: {e}")

if not os.path.exists(seq2seq_checkpoint):
    print(f"⚠️  {seq2seq_checkpoint} not found. Skipping seq2seq load.")
else:
    try:
        s2s_model_infer.load_state_dict(
            torch.load(seq2seq_checkpoint, map_location=safe_device)
        )
        print(f"✓ {seq2seq_checkpoint} loaded")
    except RuntimeError as e:
        print(f"❌ Failed to load {seq2seq_checkpoint}: {e}")

# Step 3: Load tokenizers
tokenizer_infer = None
char_tok_infer = None

if os.path.exists("detect_char_tokenizer.json"):
    try:
        tokenizer_infer = CharTokenizer.load("detect_char_tokenizer.json")
        print("✓ Detector tokenizer loaded")
    except Exception as e:
        print(f"⚠️  Failed to load detector tokenizer: {e}")
else:
    print("⚠️  detect_char_tokenizer.json not found")

if os.path.exists("seq2seq_char_tokenizer.json"):
    try:
        char_tok_infer = CharTokenizer.load("seq2seq_char_tokenizer.json")
        print("✓ Seq2Seq tokenizer loaded")
    except Exception as e:
        print(f"⚠️  Failed to load seq2seq tokenizer: {e}")
else:
    print("⚠️  seq2seq_char_tokenizer.json not found")

# Step 4: Move to GPU
device_infer = torch.device("cuda" if torch.cuda.is_available() else "cpu")
detector_infer = detector_infer.to(device_infer)
s2s_model_infer = s2s_model_infer.to(device_infer)
print(f"\n✓ Models moved to {device_infer}")
print(f"✓ Ready for inference!")

Step 1: Loading models on CPU...
❌ Failed to load detector_best.pth: PytorchStreamReader failed locating file data/0: file not found. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.
❌ Failed to load seq2seq_best copy.pth: Error(s) in loading state_dict for Seq2SeqCorrector:
	size mismatch for encoder.embedding.weight: copying a param with shape torch.Size([82, 128]) from checkpoint, the shape in current model is torch.Size([60, 128]).
	size mismatch for decoder.embedding.weight: copying a param with shape torch.Size([82, 128]) from checkpoint, the shape in current model is torch.Size([60, 128]).
	size mismatch for decoder.fc_out.weight: copying a param with shape torch.Size([82, 384]) from checkpoint, the shape in current model is torch.Size([60, 384]).
	size mismatch for decoder.f

In [15]:
def correct_word_beam(wrong_word, model, tokenizer, max_len=30, beam_width=5):
    """
    Beam search decoding — returns top-3 candidates with confidence scores.
    """
    if tokenizer is None:
        print("❌ Tokenizer not loaded!")
        return []
    
    model.eval()
    src = torch.tensor(
        [tokenizer.encode(str(wrong_word), max_len)], dtype=torch.long
    ).to(device_infer)

    SOS = tokenizer.char2idx[tokenizer.SOS]
    EOS = tokenizer.char2idx[tokenizer.EOS]
    PAD = tokenizer.char2idx[tokenizer.PAD]

    with torch.no_grad():
        enc_out, h, c = model.encoder(src)
    h = h.squeeze(0)
    c = c.squeeze(0)

    beams     = [(0.0, [], h, c)]
    completed = []

    for _ in range(max_len):
        if not beams:
            break
        candidates = []
        for lp, tokens, bh, bc in beams:
            if tokens and tokens[-1] == EOS:
                completed.append((lp, tokens))
                continue
            last = torch.tensor(
                [tokens[-1] if tokens else SOS], dtype=torch.long
            ).to(device_infer)
            with torch.no_grad():
                pred, new_h, new_c, _ = model.decoder.forward_step(
                    last, bh, bc, enc_out
                )
            log_p = torch.log_softmax(pred[0], dim=-1)
            topk_lp, topk_idx = log_p.topk(beam_width)
            for tlp, tidx in zip(topk_lp.tolist(), topk_idx.tolist()):
                candidates.append((lp + tlp, tokens + [tidx], new_h, new_c))
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]

    for lp, tokens, _, _ in beams:
        completed.append((lp, tokens))

    completed.sort(key=lambda x: x[0], reverse=True)

    def decode_tokens(tokens):
        out = []
        for idx in tokens:
            ch = tokenizer.idx2char.get(idx, tokenizer.UNK)
            if ch == tokenizer.EOS:
                break
            if ch not in (tokenizer.PAD, tokenizer.SOS):
                out.append(ch)
        return "".join(out)

    seen, results = set(), []
    for lp, tokens in completed:
        word = decode_tokens(tokens)
        if word and word not in seen:
            seen.add(word)
            norm_score = math.exp(lp / max(len(tokens), 1))
            results.append((word, round(norm_score, 4)))
        if len(results) == 3:
            break

    return results


def predict_word(text, tokenizer, max_len=30):
    """
    Detect if word is correct.
    Returns: {"text": word, "is_correct": bool, "confidence": float}
    """
    if tokenizer is None:
        return {"text": text, "is_correct": True, "confidence": 0.0}
    
    detector_infer.eval()
    indices = tokenizer.encode(text, max_len)
    x = torch.tensor([indices], dtype=torch.long).to(device_infer)
    
    with torch.no_grad():
        prob = detector_infer(x).item()
    
    return {
        "text": text,
        "is_correct": prob > 0.5,
        "confidence": round(prob, 4)
    }


def give_suggestions(word, threshold=0.5, beam_width=5):
    """
    Full pipeline: detect if wrong, then get suggestions.
    """
    result = predict_word(word, tokenizer_infer, MAX_SEQ_LEN)
    
    if not result["is_correct"] and result["confidence"] <= threshold:
        result["suggestions"] = correct_word_beam(
            word, s2s_model_infer, char_tok_infer,
            max_len=20, beam_width=beam_width
        )
    else:
        result["suggestions"] = []
    
    return result


print("✓ Inference functions defined")

✓ Inference functions defined


In [16]:
# ============================================================================
# TEST INFERENCE
# ============================================================================

if tokenizer_infer and char_tok_infer:
    print("\n" + "="*70)
    print("TESTING WORD DETECTION & CORRECTION")
    print("="*70 + "\n")
    
    test_words = [
        ("यसरी", True),      # Correct
        ("ीसरय", False),     # Wrong (scrambled)
        ("नेपालमा", True),   # Correct
        ("नपेामला", False),  # Wrong
    ]
    
    for word, expected_correct in test_words:
        result = give_suggestions(word, threshold=0.5, beam_width=5)
        
        status = "✓" if result["is_correct"] else "✗"
        print(f"{status} Word: {word}")
        print(f"  Confidence: {result['confidence']}")
        
        if result["suggestions"]:
            print(f"  Suggestions:")
            for i, (sugg, score) in enumerate(result["suggestions"], 1):
                print(f"    {i}. {sugg} (score: {score})")
        print()
else:
    print("❌ Cannot run inference: Tokenizers not loaded")


TESTING WORD DETECTION & CORRECTION

✗ Word: यसरी
  Confidence: 0.377
  Suggestions:
    1. छकँँँषषषषषषषषषषषषषषष (score: 0.0699)
    2. छकँँँझषषषषषषषषषषषषषष (score: 0.0696)
    3. छकँँझषषषषषषषषषषषषषषष (score: 0.0689)

✗ Word: ीसरय
  Confidence: 0.3807
  Suggestions:
    1. छकँँषषषषषषषषषषषषषषषष (score: 0.0862)
    2. छकँँझषषषषषषषषषषषषषषष (score: 0.0861)
    3. छकँँषषषषषषषषषषषषषषषद (score: 0.0832)



/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1,0,0], thread: [0,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1,0,0], thread: [1,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1,0,0], thread: [2,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1,0,0], thread: [3,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1,0,0], thread: [4,0,0] Assertion `ind 

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [17]:
# ============================================================================
# FULL SENTENCE CORRECTION
# ============================================================================

def correct_sentence(sentence, threshold=0.5, beam_width=5):
    """
    Correct all wrong words in a sentence.
    """
    if tokenizer_infer is None:
        return {"input": sentence, "output": sentence, "details": []}
    
    words = sentence.split()
    details = []
    corrected = []
    
    for word in words:
        result = give_suggestions(word, threshold=threshold, beam_width=beam_width)
        
        if result["suggestions"]:
            best_suggestion = result["suggestions"][0][0]
            corrected.append(best_suggestion)
        else:
            corrected.append(word)
        
        details.append(result)
    
    return {
        "input": sentence,
        "output": " ".join(corrected),
        "details": details
    }


if tokenizer_infer and char_tok_infer:
    print("\n" + "="*70)
    print("SENTENCE CORRECTION")
    print("="*70 + "\n")
    
    test_sentences = [
        "यसरी नेपालमा वस्तु बिक्री हुने",
        "ीसरय नपेामला वतु्स पिबी नेहु",
    ]
    
    for sent in test_sentences:
        result = correct_sentence(sent, threshold=0.5, beam_width=5)
        print(f"Input:  {result['input']}")
        print(f"Output: {result['output']}")
        print()
else:
    print("⚠️  Tokenizers not loaded - skipping sentence test")


SENTENCE CORRECTION



AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
